## Taxi Data

### Setup   

In [2]:
# libraries
import matplotlib.pyplot as plt ## for basic plotting
import matplotlib as mpl ## for setting default parameters
import pandas as pd
import os
from pathlib import Path

In [4]:
# paths
base_dir = Path("/Users/hannahmaihojgaard/Documents/GitHub/datascience2026")
data_path = Path(base_dir / "data")

### Load in data

In [7]:
df = pd.read_parquet(data_path / "yellow_tripdata_2026-01.parquet")
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


### Preproccessing

In [8]:
df.shape
df.columns
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


- *VendorID*: Taxi technology provider (1 = CMT, 2 = VeriFone)
- *tpep_pickup_datetime*: Timestamp when the trip started (meter on)
- *tpep_dropoff_datetime*: Timestamp when the trip ended (meter off)
- *passenger_count*: Number of passengers (driver-reported)
- *trip_distance*: Trip distance in miles (from taximeter)
- *RatecodeID*: Rate type (e.g., standard, JFK, Newark, etc.)
- *store_and_fwd_flag*: Whether trip data was stored before sending (Y/N)
- *PULocationID*: Pickup location ID (zone-based)
- *DOLocationID*: Dropoff location ID (zone-based)
- *payment_type*: Payment method (1=card, 2=cash, etc.)
- *fare_amount*: Base fare (time + distance)
- *extra*: Additional charges (e.g., rush hour, overnight)
- *mta_tax*: Fixed $0.50 MTA tax
- *tip_amount*: Tip (only recorded for card payments)
- *tolls_amount*: Total toll charges
- *improvement_surcharge*: $0.30 improvement fee
- *total_amount*: Total charged (excluding cash tips)
- *congestion_surcharge*: Fee for trips in congestion zones
- *Airport_fee*: Extra fee for airport trips
- *cbd_congestion_fee*: Central business district congestion fee


In [10]:
df.isna().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [13]:
# converting pickup and dropoff datetime columns to datetime format
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# calculating trip duration in minutes
df["trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60  # minutes 

In [ ]:
# checking for negative or zero values in trip_distance, fare_amount, and trip_duration
df[df["trip_distance"] <= 0].shape
df[df["fare_amount"] <= 0].shape
df[df["trip_duration"] <= 0].shape

(45070, 21)

In [16]:
df[df["trip_distance"] <= 0].head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration
118,1,2026-01-01 00:21:36,2026-01-01 00:22:52,1.0,0.0,1.0,N,239,43,2,...,3.50,0.5,0.0,0.0,1.0,8.00,2.5,0.0,0.00,1.266667
329,1,2026-01-01 00:11:22,2026-01-01 00:34:04,1.0,0.0,1.0,N,43,170,1,...,4.25,0.5,0.0,0.0,1.0,24.15,2.5,0.0,0.75,22.700000
330,1,2026-01-01 00:54:54,2026-01-01 01:09:08,1.0,0.0,1.0,N,161,137,1,...,4.25,0.5,4.6,0.0,1.0,23.15,2.5,0.0,0.75,14.233333
360,2,2026-01-01 00:32:10,2026-01-01 00:32:14,4.0,0.0,5.0,N,162,162,1,...,0.00,0.0,0.0,0.0,1.0,54.25,2.5,0.0,0.75,0.066667
634,2,2026-01-01 00:45:48,2026-01-01 00:45:54,1.0,0.0,5.0,N,261,261,1,...,0.00,0.0,0.0,0.0,1.0,54.25,2.5,0.0,0.75,0.100000
